# Notebook 05: City-Wide Pipeline Evaluation
**Multi-Camera Tracking System — CityFlowV2 (Vehicles) + WILDTRACK (People)**

Runs the full 7-stage MTMC pipeline on two city-scale datasets.
Downloads go to `/tmp` (larger disk), results stay in `/kaggle/working` (downloadable).

| Dataset | Objects | Cameras | Resolution |
|---------|---------|---------|------------|
| WILDTRACK | People | 7 overlapping | 1920×1080 |
| CityFlowV2 | Vehicles (880 IDs) | 46 at 16 intersections | 960p+ |

## 0. Environment Setup

In [ ]:
import os, sys, subprocess, shutil
from pathlib import Path

REPO_URL = "https://github.com/MRKDaGods/gp.git"

WORK_DIR = Path("/kaggle/working")
PROJECT_DIR = WORK_DIR / "gp"
# Use /tmp for large dataset downloads — root fs has ~50 GB vs /kaggle/working ~20 GB
DATA_DIR = Path("/tmp/mtmc_data")
MODELS_DIR = PROJECT_DIR / "models"

print(f"GPU: {subprocess.run(['nvidia-smi', '-L'], capture_output=True, text=True).stdout.strip()}")

# Show disk space on all relevant partitions
for mount in ["/kaggle/working", "/tmp", "/"]:
    try:
        t, u, f = shutil.disk_usage(mount)
        print(f"{mount}: {f / (1024**3):.1f} GB free / {t / (1024**3):.1f} GB total")
    except Exception:
        pass

In [ ]:
# Clone the repo
if PROJECT_DIR.exists():
    print(f"Project exists, pulling latest...")
    subprocess.run(["git", "-C", str(PROJECT_DIR), "pull"], check=True)
else:
    print(f"Cloning {REPO_URL}...")
    subprocess.run(["git", "clone", REPO_URL, str(PROJECT_DIR)], check=True)

os.chdir(PROJECT_DIR)
sys.path.insert(0, str(PROJECT_DIR))

# Create data directories and symlink so pipeline finds data at expected path
DATA_DIR.mkdir(parents=True, exist_ok=True)
pipeline_data_raw = PROJECT_DIR / "data" / "raw"
pipeline_data_raw.parent.mkdir(parents=True, exist_ok=True)
if pipeline_data_raw.exists() and not pipeline_data_raw.is_symlink():
    shutil.rmtree(pipeline_data_raw)
if not pipeline_data_raw.exists():
    pipeline_data_raw.symlink_to(DATA_DIR)

print(f"Working dir: {os.getcwd()}")
print(f"Data dir (symlinked): {pipeline_data_raw} -> {DATA_DIR}")
print(f"Contents: {sorted(p.name for p in PROJECT_DIR.iterdir())}")

In [ ]:
# Install dependencies — Kaggle P100 images have most of these already
!pip install -q ultralytics boxmot faiss-gpu omegaconf loguru networkx plotly click rich gdown 2>&1 | tail -5
# Verify key imports
import torch, cv2, numpy as np, faiss
print(f"PyTorch {torch.__version__} | CUDA {torch.cuda.is_available()} | GPU {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none'}")
print(f"OpenCV {cv2.__version__} | FAISS GPUs: {faiss.get_num_gpus()}")

In [ ]:
# Download model weights
!python scripts/download_models.py --models-dir models/

# Check weights
for name, p in [("YOLO", "detection/yolo26m.pt"), ("Person ReID", "reid/person_resnet50ibn_market1501.pth"),
                ("Vehicle ReID", "reid/vehicle_resnet50ibn_veri776.pth"), ("Tracker", "tracker/osnet_x0_25_msmt17.pt")]:
    fp = MODELS_DIR / p
    s = f"{fp.stat().st_size/(1024**2):.0f}MB" if fp.exists() else "MISSING"
    print(f"  {name}: {s}")

---
## 1. WILDTRACK (People, 7 cameras, 1080p)

Download ~6.3 GB, run full pipeline, package results.

In [ ]:
# Download WILDTRACK
!python scripts/download_datasets.py --dataset wildtrack

# Prepare ground truth
!python scripts/prepare_dataset.py --dataset wildtrack --root data/raw/wildtrack

In [ ]:
%%time
# Run full pipeline on WILDTRACK
!python scripts/run_pipeline.py \
    --config configs/default.yaml \
    --dataset-config configs/datasets/wildtrack.yaml \
    -o project.run_name=wildtrack_eval \
    -o stage0.input_dir=data/raw/wildtrack/videos

In [ ]:
import json

wt_output = PROJECT_DIR / "data" / "outputs" / "wildtrack_eval"

# Show trajectory results
for stage_file in ["stage4/global_trajectories.json", "stage5/evaluation_results.json"]:
    fp = wt_output / stage_file
    if fp.exists():
        with open(fp) as f:
            data = json.load(f)
        print(f"\n=== {stage_file} ===")
        if "trajectories" in stage_file or "global" in stage_file:
            print(f"Global identities: {len(data)}")
            items = list(data.items()) if isinstance(data, dict) else list(enumerate(data))
            for gid, traj in sorted(items, key=lambda x: len(x[1]) if isinstance(x[1], list) else len(x[1].get('tracklets', x[1].get('tracklet_ids', []))), reverse=True)[:10]:
                if isinstance(traj, dict):
                    tls = traj.get('tracklets', traj.get('tracklet_ids', []))
                    cams = set(t.get('camera_id','?') for t in tls if isinstance(t, dict))
                    print(f"  GID {gid}: {len(tls)} tracklets, {len(cams)} cameras")
        else:
            for k, v in data.items():
                print(f"  {k}: {v:.4f}" if isinstance(v, float) else f"  {k}: {v}")
    else:
        print(f"{stage_file}: not found")

# List all output files
print("\n=== Output files ===")
if wt_output.exists():
    for p in sorted(wt_output.rglob("*")):
        if p.is_file():
            print(f"  {p.relative_to(wt_output)} ({p.stat().st_size/(1024**2):.1f} MB)")

In [ ]:
# Free disk: delete WILDTRACK raw data, keep only pipeline outputs
wt_raw = DATA_DIR / "wildtrack"
if wt_raw.exists():
    size = sum(f.stat().st_size for f in wt_raw.rglob("*") if f.is_file()) / (1024**3)
    shutil.rmtree(wt_raw)
    print(f"Freed {size:.1f} GB by removing WILDTRACK raw data")

t, u, f = shutil.disk_usage("/tmp")
print(f"/tmp: {f/(1024**3):.1f} GB free")

---
## 2. CityFlowV2 (Vehicles, 46 cameras, city intersections)

Download ~15–20 GB from Google Drive, run full pipeline.

In [ ]:
# Download CityFlowV2
!python scripts/download_datasets.py --dataset cityflowv2

# Prepare dataset structure
!python scripts/prepare_dataset.py --dataset cityflowv2 --root data/raw/cityflowv2

In [ ]:
%%time
# Run full pipeline on CityFlowV2
!python scripts/run_pipeline.py \
    --config configs/default.yaml \
    --dataset-config configs/datasets/cityflowv2.yaml \
    -o project.run_name=cityflowv2_eval \
    -o stage0.input_dir=data/raw/cityflowv2

In [ ]:
cfv2_output = PROJECT_DIR / "data" / "outputs" / "cityflowv2_eval"

for stage_file in ["stage4/global_trajectories.json", "stage5/evaluation_results.json"]:
    fp = cfv2_output / stage_file
    if fp.exists():
        with open(fp) as f:
            data = json.load(f)
        print(f"\n=== {stage_file} ===")
        if "trajectories" in stage_file or "global" in stage_file:
            print(f"Global identities: {len(data)}")
            items = list(data.items()) if isinstance(data, dict) else list(enumerate(data))
            for gid, traj in sorted(items, key=lambda x: len(x[1]) if isinstance(x[1], list) else len(x[1].get('tracklets', x[1].get('tracklet_ids', []))), reverse=True)[:10]:
                if isinstance(traj, dict):
                    tls = traj.get('tracklets', traj.get('tracklet_ids', []))
                    cams = set(t.get('camera_id','?') for t in tls if isinstance(t, dict))
                    print(f"  GID {gid}: {len(tls)} tracklets, {len(cams)} cameras")
        else:
            for k, v in data.items():
                print(f"  {k}: {v:.4f}" if isinstance(v, float) else f"  {k}: {v}")
    else:
        print(f"{stage_file}: not found")

print("\n=== Output files ===")
if cfv2_output.exists():
    for p in sorted(cfv2_output.rglob("*")):
        if p.is_file():
            print(f"  {p.relative_to(cfv2_output)} ({p.stat().st_size/(1024**2):.1f} MB)")

---
## 3. Comparison & Visualization

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

def load_run_summary(run_dir, name):
    s = {"dataset": name}
    tp = run_dir / "stage4" / "global_trajectories.json"
    if tp.exists():
        with open(tp) as f:
            trajs = json.load(f)
        s["global_ids"] = len(trajs)
        all_tl, cams = [], set()
        for t in (trajs.values() if isinstance(trajs, dict) else trajs):
            tl = t.get("tracklets", t.get("tracklet_ids", [])) if isinstance(t, dict) else []
            all_tl.extend(tl)
            for tr in tl:
                if isinstance(tr, dict): cams.add(tr.get("camera_id","?"))
        s["tracklets"] = len(all_tl)
        s["cameras"] = len(cams)
    ep = run_dir / "stage5" / "evaluation_results.json"
    if ep.exists():
        with open(ep) as f:
            m = json.load(f)
        for k in ["HOTA","MOTA","IDF1"]: 
            if k in m: s[k] = m[k]
    return s

rows = []
for name, d in [("WILDTRACK", wt_output), ("CityFlowV2", cfv2_output)]:
    if d.exists(): rows.append(load_run_summary(d, name))
if rows:
    df = pd.DataFrame(rows).set_index("dataset")
    print(df.to_string())
else:
    print("No results yet.")

In [ ]:
# Show sample annotated frames from each dataset
def show_frames(run_dir, name, n=3):
    vid_dir = run_dir / "stage6" / "videos"
    if not vid_dir.exists(): 
        print(f"No videos for {name}"); return
    vids = sorted(vid_dir.glob("*.mp4"))[:4]
    if not vids: 
        print(f"No MP4s in {vid_dir}"); return
    fig, axes = plt.subplots(len(vids), n, figsize=(5*n, 4*len(vids)))
    if len(vids) == 1: axes = [axes]
    for row, vp in enumerate(vids):
        cap = cv2.VideoCapture(str(vp))
        total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        for col, fi in enumerate([int(total*i/(n+1)) for i in range(1,n+1)]):
            cap.set(cv2.CAP_PROP_POS_FRAMES, fi)
            ret, frame = cap.read()
            ax = axes[row][col] if n > 1 else axes[row]
            if ret: ax.imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
            ax.set_title(f"{vp.stem} f{fi}", fontsize=9); ax.axis("off")
        cap.release()
    plt.suptitle(f"{name} — Annotated Frames", fontsize=14, fontweight="bold")
    plt.tight_layout(); plt.show()

show_frames(wt_output, "WILDTRACK")
show_frames(cfv2_output, "CityFlowV2")

---
## 4. Package Results for Download

In [ ]:
import zipfile

def package(run_dir, name):
    zp = WORK_DIR / f"{name}_results.zip"
    if not run_dir.exists(): print(f"{name}: no results"); return
    patterns = ["stage4/*.json", "stage5/*.json", "stage5/*.html",
                "stage6/videos/*.mp4", "stage6/*.html", "stage6/exports/*",
                "config.yaml", "pipeline.log"]
    with zipfile.ZipFile(zp, "w", zipfile.ZIP_DEFLATED) as zf:
        for pat in patterns:
            for f in run_dir.glob(pat):
                if f.is_file():
                    zf.write(f, f"{name}/{f.relative_to(run_dir)}")
    print(f"{name}: {zp.name} ({zp.stat().st_size/(1024**2):.1f} MB)")

package(wt_output, "wildtrack")
package(cfv2_output, "cityflowv2")
print("\nDownload from the Output tab.")